1. Setup and Video Initialization

In [35]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Define paths to your specific project videos 
video_dir = 'data/videos/'
video_files = ['project_video.mp4', 'challenge_video.mp4', 'harder_challenge_video.mp4']
video_file = video_files[1]
current_video = os.path.join(video_dir, video_file)

cap = cv2.VideoCapture(current_video)
print(f"SUCCESS: Loaded {video_file} at {int(cap.get(cv2.CAP_PROP_FPS))} FPS")

SUCCESS: Loaded challenge_video.mp4 at 29 FPS


2. Pre-processing Functions

convert to grayscale and apply Gaussian smoothing to reduce noise before edge detection.

In [36]:
def preprocess_frame(frame):
    # 1. Convert to HLS color space
    hls = cv2.cvtColor(frame, cv2.COLOR_RGB2HLS)
    
    # 2. Define White Lane Mask (High Lightness)
    lower_white = np.array([0, 200, 0])
    upper_white = np.array([180, 255, 255])
    white_mask = cv2.inRange(hls, lower_white, upper_white)
    
    # 3. Define Yellow Lane Mask (Specific Hue + High Saturation)
    lower_yellow = np.array([15, 0, 100])
    upper_yellow = np.array([35, 255, 255])
    yellow_mask = cv2.inRange(hls, lower_yellow, upper_yellow)
    
    # 4. Combine masks and apply to original image or grayscale
    combined_mask = cv2.bitwise_or(white_mask, yellow_mask)
    
    # Apply mask to grayscale to maintain edge detection compatibility
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    masked_gray = cv2.bitwise_and(gray, combined_mask)
    
    # 5. Apply Gaussian smoothing
    kernel_size = 5
    blur_gray = cv2.GaussianBlur(masked_gray, (kernel_size, kernel_size), 0)
    
    return blur_gray

def get_roi_mask(image):
    # Define a road region-of-interest (ROI) 
    mask = np.zeros_like(image)
    imshape = image.shape
    
    # Vertices for a trapezoidal ROI covering the highway lane 
    vertices = np.array([[(0, imshape[0]), (450, 320), (500, 320), (imshape[1], imshape[0])]], dtype=np.int32)
    
    cv2.fillPoly(mask, vertices, 255)
    masked_image = cv2.bitwise_and(image, mask)
    return masked_image

3. Edge and Line Detection

This implements the Canny and Hough Transform steps to find lane candidates.

In [37]:
def detect_lanes(frame):
    # 1. Preprocess (Color Masking + Blur)
    processed = preprocess_frame(frame)
    
    # 2. Canny Edge Detection 
    # Thresholds can be lower now because the noise was reduced by the color mask
    low_threshold = 40
    high_threshold = 120
    edges = cv2.Canny(processed, low_threshold, high_threshold)
    
    # 3. Mask ROI 
    masked_edges = get_roi_mask(edges)
    
    # 4. Hough Line Transform 
    rho = 1              
    theta = np.pi/180    
    threshold = 15       # Slightly lower threshold to catch fainter segments
    min_line_len = 10    
    max_line_gap = 250   
    
    lines = cv2.HoughLinesP(masked_edges, rho, theta, threshold, np.array([]),
                            minLineLength=min_line_len, maxLineGap=max_line_gap)
    
    return lines

4. Main Processing Loop

run the pipeline

In [ ]:
# 1. Setup Video Writer 
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
output_name =  f'{video_file.split('.')[0]}_detection_v2.mp4'
# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out = cv2.VideoWriter(output_name, fourcc, fps, (frame_width, frame_height))
print("Processing video... Please wait.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # --- The Pipeline Logic ---
    # 1. Detect raw lines using your detect_lanes function
    lines = detect_lanes(frame)
    
    # 2. Create an empty image for the overlay
    line_image = np.zeros_like(frame)
    
    # 3. Draw the lines (Add your averaging/extrapolation logic here later)
    if lines is not None:
        for line in lines:
            for x1, y1, x2, y2 in line:
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 10)
    
    # 4. Merge overlay with original frame
    final_frame = cv2.addWeighted(frame, 0.8, line_image, 1, 0)
    
    # 5. Write the frame to the new file
    out.write(final_frame)

# Clean up
cap.release()
out.release()
print(f"Success! Video saved as '{output_name}'")

Processing video... Please wait.
